# 03 — Word2Vec with gensim

Runnable companion to **notes 13–15**: word embeddings, the pre-trained Google model,
training your own, CBOW vs Skip-gram, and average Word2Vec.

| Section | Note |
|---|---|
| 1. Setup | — |
| 2. The Google pre-trained model | [13](../notes/13-Word-Embeddings-and-Word2Vec.md) |
| 3. Cosine similarity from scratch | 13 |
| 4. `king - man + woman` | 13 |
| 5. Training your own | [14](../notes/14-CBOW-and-SkipGram.md) |
| 6. CBOW vs Skip-gram | 14 |
| 7. Average Word2Vec | [15](../notes/15-Average-Word2Vec.md) |
| 8. Visualising the space | — |

> **Heads-up:** section 2 downloads a **1.6 GB** model. If you are short on
> disk/bandwidth, skip to section 5 — training your own needs no download.

## 1. Setup

In [ ]:
# !pip install gensim numpy pandas matplotlib scikit-learn nltk

import numpy as np
import pandas as pd

import gensim
print("gensim", gensim.__version__)

## 2. The Google pre-trained model  ([note 13](../notes/13-Word-Embeddings-and-Word2Vec.md))

| | |
|---|---|
| Trained on | Google News, ~100 billion words |
| Vocabulary | 3 million words and phrases |
| Dimensions | **300** |
| Download | ~1.6 GB (cached after the first run) |

In [ ]:
import gensim.downloader as api

# First run downloads ~1.6 GB. Afterwards it loads from ~/gensim-data/
wv = api.load('word2vec-google-news-300')
print("loaded:", type(wv))

In [ ]:
vec_king = wv['king']

print("shape:", vec_king.shape)
print("dtype:", vec_king.dtype)
print("first 8 numbers:", vec_king[:8].round(4))

### 2.1 Dense, not sparse — count the zeros

In [ ]:
print("non-zero entries:", int(np.count_nonzero(vec_king)), "out of", vec_king.size)
print()
print("Compare with bag of words on a 50,000-word vocabulary:")
print("  BoW      : 1 non-zero out of 50,000   (99.998% zeros)")
print("  Word2Vec :", int(np.count_nonzero(vec_king)), "non-zero out of 300  (dense)")

### 2.2 Every word gets the same 300 dimensions

In [ ]:
for w in ['king', 'cricket', 'happy', 'democracy', 'pizza']:
    print(f"{w:12} -> {wv[w].shape}")

### 2.3 Finding similar words — with no dictionary or thesaurus

In [ ]:
for word in ['cricket', 'happy', 'india']:
    print(f"most similar to {word!r}:")
    for w, score in wv.most_similar(word, topn=6):
        print(f"    {w:22} {score:.4f}")
    print()

Look at the `happy` list: `glad`, `pleased`, `ecstatic`, `overjoyed`. **Genuine synonyms,
discovered purely from which words appear in similar contexts** across 100 billion words.
No human labelled any of it.

### 2.4 Similarity between two specific words

In [ ]:
pairs = [('king','queen'), ('hockey','sports'), ('king','banana'),
         ('happy','glad'), ('happy','angry'), ('paris','france')]

for a, b in pairs:
    print(f"similarity({a:8}, {b:8}) = {wv.similarity(a, b):.4f}")

Note `happy` / `angry` still scores moderately high. Word2Vec places **antonyms close
together** because they appear in the *same contexts* ("I am ___ about the result"). It
measures relatedness, not agreement — a known limitation worth remembering.

### 2.5 ❌ Out of vocabulary is reduced, not solved

In [ ]:
for w in ['cricket', 'supercalifragilistic', 'asdfghjkl']:
    try:
        wv[w]
        print(f"{w:22} -> OK")
    except KeyError:
        print(f"{w:22} -> KeyError: not in the 3M vocabulary")

Classic Word2Vec cannot build a vector for a word it never saw. **FastText** can, because
it embeds character n-grams:

```python
import gensim.downloader as api
ft = api.load('fasttext-wiki-news-subwords-300')
ft['asdfghjkl']        # works -- assembled from character n-grams
```

## 3. Cosine similarity from scratch

`1 - cos(theta)` is the distance; `cos(theta)` is the similarity.

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def cosine_distance(a, b):
    return 1 - cosine_similarity(a, b)

for a, b in [('king','queen'), ('king','banana')]:
    s = cosine_similarity(wv[a], wv[b])
    print(f"{a:6} vs {b:8}  similarity={s:.4f}  distance={1-s:.4f}  "
          f"angle={np.degrees(np.arccos(np.clip(s,-1,1))):.1f} deg")

In [ ]:
print("matches gensim?", np.isclose(cosine_similarity(wv['king'], wv['queen']),
                                    wv.similarity('king','queen')))

### 3.1 Why cosine and not Euclidean distance

In [ ]:
a = wv['king']
b = a * 10          # same DIRECTION, ten times the magnitude

print("cosine similarity(a, 10a) =", round(cosine_similarity(a, b), 6), " <- unchanged")
print("euclidean distance(a, 10a) =", round(float(np.linalg.norm(a - b)), 4), " <- huge")
print()
print("Magnitude reflects frequency more than meaning; direction carries the semantics.")

## 4. `king - man + woman`

In [ ]:
result = wv['king'] - wv['man'] + wv['woman']

print("result vector shape:", result.shape)
print()
for w, score in wv.most_similar([result], topn=6):
    print(f"    {w:18} {score:.4f}")

`king` comes first (you started there and moved only a little), but the first **new** word
is `queen`, then `monarch`, `princess`, `crown_prince`. The neighbourhood is exactly right.

gensim has a shortcut that excludes the input words for you:

In [ ]:
print(wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3))

### 4.1 Other analogies

In [ ]:
analogies = [
    (['paris', 'italy'],    ['france'],  'capital-of'),
    (['walked', 'swim'],    ['walk'],    'past tense'),
    (['bigger', 'small'],   ['big'],     'comparative'),
    (['windows', 'google'], ['microsoft'], 'company product'),
]

for pos, neg, label in analogies:
    try:
        top = wv.most_similar(positive=pos, negative=neg, topn=2)
        print(f"{label:16} {pos} - {neg} = {[t[0] for t in top]}")
    except KeyError as e:
        print(f"{label:16} skipped ({e})")

### 4.2 ⚠️ The model inherits the bias of its training data

In [ ]:
print("doctor - man + woman =", wv.most_similar(positive=['doctor','woman'],
                                                 negative=['man'], topn=3))
print("programmer - man + woman =", wv.most_similar(positive=['computer_programmer','woman'],
                                                    negative=['man'], topn=3))

These reflect statistical associations in news text, not facts. **Deploying such a model
in hiring, lending or moderation causes real harm.** Audit before you ship; consider
debiasing techniques or a curated corpus.

## 5. Training your own  ([note 14](../notes/14-CBOW-and-SkipGram.md))

No download needed from here on.

In [ ]:
from gensim.utils import simple_preprocess

raw_corpus = [
    "the king ruled the kingdom with his queen",
    "the queen and the king lived in the royal palace",
    "a prince and a princess are royal children",
    "the boy played football with his friends",
    "the girl played football with her friends",
    "boys and girls play games together every day",
    "pizza and burger are popular fast food items",
    "i love eating pizza with extra cheese",
    "burger and fries make a tasty food combination",
    "apple and mango are sweet healthy fruits",
    "she eats an apple and a mango every morning",
    "fruits like apple mango and banana are healthy food",
] * 40          # repeat so the tiny corpus has enough signal to train on

sentences = [simple_preprocess(doc) for doc in raw_corpus]
print("type:", type(sentences), "of", type(sentences[0]))
print("first:", sentences[0])

### ⚠️ The #1 beginner error: passing strings instead of token lists

In [ ]:
from gensim.models import Word2Vec

wrong = Word2Vec(raw_corpus, vector_size=20, min_count=1, epochs=1)   # STRINGS -- wrong
print("vocabulary when you pass strings:", wrong.wv.index_to_key[:15])
print("^ single CHARACTERS -- gensim iterated each string character by character")

In [ ]:
model = Word2Vec(
    sentences,
    vector_size = 100,   # embedding dimension == hidden layer size
    window      = 5,     # context window -- a SEPARATE hyperparameter
    min_count   = 2,     # ignore words appearing fewer than twice
    workers     = 4,
    sg          = 0,     # 0 = CBOW, 1 = Skip-gram
    epochs      = 50,
    seed        = 42,
)

print("vocabulary size :", len(model.wv.index_to_key))
print("documents       :", model.corpus_count)
print("epochs          :", model.epochs)
print("first 15 words  :", model.wv.index_to_key[:15])

### 5.1 `vector_size` is the dimension — `window` is NOT

In [ ]:
for vs in [50, 100, 300]:
    m = Word2Vec(sentences, vector_size=vs, window=5, min_count=2, epochs=5, seed=42)
    print(f"vector_size={vs:3}, window=5  ->  model.wv['king'].shape = {m.wv['king'].shape}")

print()
for win in [2, 5, 10]:
    m = Word2Vec(sentences, vector_size=100, window=win, min_count=2, epochs=5, seed=42)
    print(f"vector_size=100, window={win:2}  ->  model.wv['king'].shape = {m.wv['king'].shape}")
print()
print("The dimension follows vector_size ONLY. window never changes it.")

### 5.2 What the little model learned

In [ ]:
for w in ['king', 'pizza', 'apple']:
    if w in model.wv:
        print(f"most similar to {w!r}:", [x[0] for x in model.wv.most_similar(w, topn=4)])

On a 480-sentence toy corpus the results are noisy — that is the honest lesson.
**Word2Vec needs millions of words to produce good vectors.** Below that, use pre-trained.

### 5.3 Saving and loading

In [ ]:
model.save("my_word2vec.model")
reloaded = Word2Vec.load("my_word2vec.model")
print("reloaded vocabulary size:", len(reloaded.wv.index_to_key))

# portable text format (works with other tools)
model.wv.save_word2vec_format("my_vectors.txt", binary=False)
print("also written: my_vectors.txt")

### 5.4 Continuing training on new data

In [ ]:
new_sentences = [simple_preprocess("the emperor and empress ruled the empire")] * 20

model.build_vocab(new_sentences, update=True)
model.train(new_sentences, total_examples=len(new_sentences), epochs=5)
print("'emperor' in vocabulary now?", 'emperor' in model.wv)

## 6. CBOW vs Skip-gram  ([note 14](../notes/14-CBOW-and-SkipGram.md))

### 6.1 Building the training pairs by hand

In [ ]:
def make_pairs(tokens, window=5):
    """Slide an odd-sized window; centre word is the target, the rest is context."""
    half, rows = window // 2, []
    for i in range(half, len(tokens) - half):
        context = tokens[i-half:i] + tokens[i+1:i+half+1]
        rows.append((context, tokens[i]))
    return rows

toy = "ineuron company is related to data science".split()
print("corpus:", toy)
print()
print("CBOW   (context -> target)")
for ctx, tgt in make_pairs(toy, 5):
    print(f"   X = {str(ctx):50}  y = {tgt}")

print()
print("SKIP-GRAM (target -> context)  -- same windows, reversed")
for ctx, tgt in make_pairs(toy, 5):
    print(f"   X = {tgt:10}  y = {ctx}")

Notice Skip-gram produces **4 training examples per window** where CBOW produces 1 — which
is why it is slower, and why rare words get more gradient signal.

### 6.2 One-hot encoding the vocabulary (the network's actual input)

In [ ]:
vocab_toy = list(dict.fromkeys(toy))         # preserve order, drop duplicates
V = len(vocab_toy)

onehot = pd.DataFrame(np.eye(V, dtype=int), index=vocab_toy, columns=vocab_toy)
print(f"vocabulary size V = {V}")
onehot

Because the input is one-hot, multiplying by the weight matrix `W` **just selects one row
of `W`** — and that row is the word's embedding. That is the whole trick of note 14.

### 6.3 Training both and comparing

In [ ]:
import time

for sg, name in [(0, 'CBOW'), (1, 'Skip-gram')]:
    t = time.time()
    m = Word2Vec(sentences, vector_size=100, window=5, min_count=2,
                 sg=sg, epochs=30, seed=42, workers=1)
    elapsed = time.time() - t
    sim = m.wv.similarity('king', 'queen') if 'king' in m.wv and 'queen' in m.wv else float('nan')
    print(f"{name:10} trained in {elapsed:.2f}s   similarity(king, queen) = {sim:.4f}")

```
   Small corpus, want speed, common words matter  -->  CBOW      (sg=0, default)
   Large corpus, rare/technical words matter      -->  SKIP-GRAM (sg=1)
```

## 7. Average Word2Vec  ([note 15](../notes/15-Average-Word2Vec.md))

### 7.1 The problem: one vector per WORD, but classifiers need one per DOCUMENT

In [ ]:
sentence = "the king ruled the kingdom"
tokens   = simple_preprocess(sentence)

per_word = np.array([model.wv[w] for w in tokens if w in model.wv])
print("tokens:", tokens)
print("per-word matrix shape:", per_word.shape, " <- variable number of rows")

for s in ["the king ruled the kingdom", "pizza", "boys and girls play games together every day"]:
    t = [w for w in simple_preprocess(s) if w in model.wv]
    print(f"  {len(t):2} known words -> matrix {np.array([model.wv[w] for w in t]).shape}")

### 7.2 The fix: average down the columns

In [ ]:
def avg_word2vec(doc, m):
    """doc: list of tokens. Returns ONE vector = the mean of known word vectors."""
    vecs = [m.wv[w] for w in doc if w in m.wv.index_to_key]
    if len(vecs) == 0:
        return np.zeros(m.vector_size)      # <- guard: np.mean([]) is NaN
    return np.mean(vecs, axis=0)            # <- axis=0 averages DOWN the columns

for s in ["the king ruled the kingdom", "pizza", "boys and girls play games together every day"]:
    v = avg_word2vec(simple_preprocess(s), model)
    print(f"{len(simple_preprocess(s)):2} words -> {v.shape}   {s!r}")
print()
print("Always the same length. THAT is the point.")

### 7.3 ⚠️ `axis=0` vs `axis=1` — a silent, unraised bug

In [ ]:
m = np.array([[0.2, 0.5, 0.1],
              [0.8, 0.3, 0.6],
              [0.1, 0.4, 0.2]])

print("axis=0 (correct)  :", np.mean(m, axis=0), "  shape", np.mean(m, axis=0).shape,
      "-> one value per DIMENSION")
print("axis=1 (wrong)    :", np.mean(m, axis=1), "  shape", np.mean(m, axis=1).shape,
      "-> one value per WORD")
print()
print("axis=1 raises no error. It just silently produces the wrong features.")

### 7.4 ⚠️ Empty documents produce NaN

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("np.mean([], axis=0) =", np.mean([], axis=0))

print()
print("unguarded:", end=" ")
try:
    bad = np.mean([model.wv[w] for w in [] if w in model.wv], axis=0)
    print(bad)
except Exception as e:
    print(type(e).__name__, e)

print("guarded  :", avg_word2vec([], model)[:5], "... (zeros)")

A single `NaN` row surfaces much later as
`ValueError: Input contains NaN, infinity or a value too large` at `fit()` time — far from
the real cause. Guard at the source.

### 7.5 Building the feature matrix (the fast way)

In [ ]:
docs = [simple_preprocess(d) for d in raw_corpus[:12]]

# SLOW: pd.concat in a loop is O(n^2) -- minutes on 5,000 rows
# df = pd.DataFrame()
# for i in range(len(X)):
#     df = pd.concat([df, pd.DataFrame(X[i].reshape(1, -1))], ignore_index=True)

# FAST: one vstack
X = np.vstack([avg_word2vec(d, model) for d in docs])
print("X.shape =", X.shape)

df = pd.DataFrame(X)
df.head(3).iloc[:, :6].round(4)

### 7.6 Why `.reshape(1, -1)` appears in so much tutorial code

In [ ]:
v = avg_word2vec(docs[0], model)
print("a single document vector :", v.shape, " <- 1-D, a DataFrame cannot take this as a row")
print("after reshape(1, -1)     :", v.reshape(1, -1).shape, " <- 1 row, 100 columns")
print("(-1 means: work the remaining dimension out yourself)")

### 7.7 Semantic similarity survives the averaging — the real win

In [ ]:
pairs = [("the king ruled the kingdom", "the queen and the king lived in the royal palace"),
         ("the king ruled the kingdom", "pizza and burger are popular fast food items")]

for a, b in pairs:
    va = avg_word2vec(simple_preprocess(a), model)
    vb = avg_word2vec(simple_preprocess(b), model)
    print(f"cos = {cosine_similarity(va, vb):+.4f}")
    print(f"   A: {a}")
    print(f"   B: {b}")
    print()

### 7.8 ❌ What averaging does NOT fix: negation and word order

In [ ]:
a = avg_word2vec(simple_preprocess("the food is good"), model)
b = avg_word2vec(simple_preprocess("the food is not good"), model)
print("cos('good' vs 'not good') =", round(cosine_similarity(a, b), 4))
print("Still nearly identical -- averaging is order-blind by construction.")
print()
print("Mitigations: keep negations out of your stopword list, use TF-IDF-weighted")
print("averaging, or move to an RNN/Transformer that reads the sequence.")

### 7.9 A better alternative: TF-IDF-weighted averaging

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus_strings = [' '.join(d) for d in docs]
tfidf = TfidfVectorizer().fit(corpus_strings)
idf_of = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
default_idf = float(max(tfidf.idf_))

def tfidf_weighted_w2v(doc, m):
    vecs, weights = [], []
    for w in doc:
        if w in m.wv.index_to_key:
            vecs.append(m.wv[w])
            weights.append(idf_of.get(w, default_idf))
    if not vecs:
        return np.zeros(m.vector_size)
    return np.average(vecs, axis=0, weights=weights)

plain    = avg_word2vec(docs[0], model)
weighted = tfidf_weighted_w2v(docs[0], model)
print("plain mean    :", plain[:5].round(4))
print("tfidf-weighted:", weighted[:5].round(4))
print("cosine between them:", round(cosine_similarity(plain, weighted), 4))

## 8. Visualising the embedding space

300 (or 100) dimensions cannot be drawn, so project to 2-D with PCA.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# use the pre-trained model if it loaded; otherwise fall back to the toy model
try:
    space, words_plot = wv, ['king','queen','prince','princess','man','woman','boy','girl',
                             'pizza','burger','apple','mango','food','cricket','football']
    vectors = np.array([space[w] for w in words_plot])
except NameError:
    space = model.wv
    words_plot = [w for w in ['king','queen','prince','princess','boy','girl',
                              'pizza','burger','apple','mango','food','football']
                  if w in space]
    vectors = np.array([space[w] for w in words_plot])

xy = PCA(n_components=2, random_state=42).fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(xy[:, 0], xy[:, 1], s=60)
for i, w in enumerate(words_plot):
    plt.annotate(w, (xy[i, 0], xy[i, 1]), fontsize=11,
                 xytext=(5, 5), textcoords='offset points')
plt.title("Word embeddings projected to 2-D with PCA")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Royalty clusters together, food clusters together, sports cluster together — **and nobody
told the model what any of those categories are.** It learned them from word co-occurrence.

> PCA discards most of the variance, so treat the picture as a sketch. For nicer clusters try
> t-SNE (`sklearn.manifold.TSNE`) or UMAP.

---

## Exercises

1. Find five analogy triples that work and five that fail. What do the failures have in common?
2. Train Word2Vec on a real corpus (20 Newsgroups, or your own text) with `vector_size` in
   `[50, 100, 300]` and compare `most_similar` quality.
3. Compare `window=2` against `window=15` on the same corpus. Does small-window similarity
   feel more *syntactic* and large-window more *topical*?
4. Implement `most_similar` yourself with plain NumPy and check it against gensim.
5. Compare plain averaging with TF-IDF-weighted averaging as features for a classifier — does
   the weighting help?
6. Load `fasttext-wiki-news-subwords-300` and check it produces a vector for a made-up word.

**Next notebook:** [`04-spam-ham-project.ipynb`](04-spam-ham-project.ipynb) — all three
vectorization strategies compared on real data.